In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
import spacy

from sklearn.decomposition import TruncatedSVD
import numpy as np
import seaborn as sns

In [2]:
df = pd.read_csv(r'C:\Users\LN\Documents\Jedha\Projets\NORTH_FACE\Helene\data.csv')

In [3]:
type(df)

pandas.core.frame.DataFrame

In [4]:
df.head()

,id,description
0,1,Active classic boxers - There's a reason why o...
1,2,Active sport boxer briefs - Skinning up Glory ...
2,3,Active sport briefs - These superbreathable no...
3,4,"Alpine guide pants - Skin in, climb ice, switc..."
4,5,"Alpine wind jkt - On high ridges, steep ice an..."


In [5]:
df["raw"] = df["description"].reset_index(drop=True)

In [6]:
df[["label", "description"]] = df["description"].str.split(
    r"\s+-\s+", n=1, expand=True
).reset_index(drop=True)

In [7]:
# Lignes dont le label apparaît plusieurs fois
duplicate_labels = df[df.duplicated(subset="label", keep=False)]
display(duplicate_labels.sort_values("label"))

# Lignes dont la description apparaît plusieurs fois
duplicate_description = df[df.duplicated(subset="description", keep=False)]
display(duplicate_description.sort_values("description"))

,id,description,raw,label
492,493,Patagonia's timeless '73 Logo T-Shirt pays hom...,'73 logo t-shirt - Patagonia's timeless '73 Lo...,'73 logo t-shirt
57,58,Soft-wearing ringspun organic cotton; Screen-p...,'73 logo t-shirt - Soft-wearing ringspun organ...,'73 logo t-shirt
494,495,"These featherweight, quick-wicking briefs keep...","Active briefs - These featherweight, quick-wic...",Active briefs
299,300,Whether you're beating the heat in Bali or ski...,Active briefs - Whether you're beating the hea...,Active briefs
497,498,No need to use that morning Times as an umbrel...,All-time shell - No need to use that morning T...,All-time shell
...,...,...,...,...
425,426,Mom and Dad may remember when hats like this k...,Trucker hat - Mom and Dad may remember when ha...,Trucker hat
152,153,"long - Blackberries, barbwire, basalt...the fa...","Watermaster waders - long - Blackberries, barb...",Watermaster waders
153,154,"short - Blackberries, barbwire, basalt...the f...","Watermaster waders - short - Blackberries, bar...",Watermaster waders
354,355,"king - Blackberries, barbwire, basalt...the fa...","Watermaster waders - king - Blackberries, barb...",Watermaster waders


,id,description,raw,label
243,244,"Our fixed triangle bikini top, the Kupala, has...",Print kupala top - Our fixed triangle bikini t...,Print kupala top
252,253,"Our fixed triangle bikini top, the Kupala, has...",Solid kupala top - Our fixed triangle bikini t...,Solid kupala top
57,58,Soft-wearing ringspun organic cotton; Screen-p...,'73 logo t-shirt - Soft-wearing ringspun organ...,'73 logo t-shirt
62,63,Soft-wearing ringspun organic cotton; Screen-p...,Flying fish t-shirt - Soft-wearing ringspun or...,Flying fish t-shirt
63,64,Soft-wearing ringspun organic cotton; Screen-p...,Gpiw classic t-shirt - Soft-wearing ringspun o...,Gpiw classic t-shirt
64,65,Soft-wearing ringspun organic cotton; Screen-p...,Live simply guitar t-shirt - Soft-wearing ring...,Live simply guitar t-shirt
431,432,Soft-wearing ringspun organic cotton; Screen-p...,Girl's live simply deer t-shirt - Soft-wearing...,Girl's live simply deer t-shirt
432,433,Soft-wearing ringspun organic cotton; Screen-p...,Girl's live simply seal t-shirt - Soft-wearing...,Girl's live simply seal t-shirt
254,255,Stay-put Sunamees fit straight across the hips...,Solid sunamee btm - Stay-put Sunamees fit stra...,Solid sunamee btm
261,262,Stay-put Sunamees fit straight across the hips...,Print sunamee btm - Stay-put Sunamees fit stra...,Print sunamee btm


In [8]:
# load the language model
# !python -m spacy download en -q
import en_core_web_sm
nlp = spacy.load("en_core_web_sm")

In [9]:
# lowercase
df['nlp'] = df['description'].apply(lambda x: x.lower())

In [10]:
df.head(1)

,id,description,raw,label,nlp
0,1,There's a reason why our boxers are a cult fav...,Active classic boxers - There's a reason why o...,Active classic boxers,there's a reason why our boxers are a cult fav...


In [11]:
# remove punctuation # alternative df["clean_text"] = df["clean_text"].apply(lambda t: " ".join("".join(c if c.isalnum() else " " for c in t).split()))
df['nlp'] = df['nlp'].str.replace("[^A-Za-z0-9%]+", " ", regex=True) # remplace les caractères qui ne sont ni des lettres ni des chiffres par des espaces. Par des défaut, la règle n'est 

In [12]:
# verify missing values
df['nlp'].isna().sum()
# Absence de données manquantes. Autrement, nous aurions utilisé la méthode .fillna(' ') pour remplir les valeurs manquantes.

np.int64(0)

In [13]:
cols = ['id', 'description', 'raw', 'label']
df = df.drop(columns=cols)

In [14]:
df.head(1)

,nlp
0,there s a reason why our boxers are a cult fav...


In [15]:
# tokenization
tokenized_doc = df['nlp'].apply(lambda x: nlp(x))

In [16]:
# apply stop words
from spacy.lang.en.stop_words import STOP_WORDS
tokenized_doc = tokenized_doc.apply(lambda x: [token.lemma_ for token in x if token.text not in STOP_WORDS])
tokenized_doc

0      [s, reason, boxer, cult, favorite, cool, espec...
1      [skin, glory, require, movement, boxer, decide...
2      [superbreathable, fly, brief, minimalist, s, c...
3      [skin, climb, ice, switch, rock, traverse, kni...
4      [high, ridge, steep, ice, alpine, jacket, serv...
                             ...                        
495    [cut, loose, madden, crowd, search, undone, ca...
496    [crew, take, edge, fickle, weather, clearing, ...
497    [need, use, morning, time, umbrella, time, han...
498    [wear, cargo, short, bask, glory, sweat, stain...
499    [time, simplify, wear, short, prove, short, co...
Name: nlp, Length: 500, dtype: object

All our documents have been tokenized and cleaned. Nlp column is ready for tfidf.

# TF IDF

In [17]:
# re inject the cleaned documents in the DataFrame.
df["nlp"] = tokenized_doc.apply(lambda x: ' '.join(x))
df

,nlp
0,s reason boxer cult favorite cool especially s...
1,skin glory require movement boxer decide poach...
2,superbreathable fly brief minimalist s choice ...
3,skin climb ice switch rock traverse knife edge...
4,high ridge steep ice alpine jacket serve true ...
...,...
495,cut loose madden crowd search undone capilene ...
496,crew take edge fickle weather clearing conditi...
497,need use morning time umbrella time handsome m...
498,wear cargo short bask glory sweat stain paint ...


In [18]:
df.to_csv(r'C:\Users\LN\Documents\Jedha\Projets\NORTH_FACE\Helene\nlp.csv')

In [19]:
# # TF-IDF vector
vectorizer = TfidfVectorizer(stop_words='english', smooth_idf=True, max_df=0.5)  # ignore les mots présents dans plus de 50 % des descriptions)
X = vectorizer.fit_transform(df['nlp'])
dense = X.toarray() # avoid risk to obtain only sparse matrix
dense

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(500, 3792))

In [20]:
vectorizer.vocabulary_

{'reason': 2647,
 'boxer': 622,
 'cult': 1003,
 'favorite': 1367,
 'cool': 929,
 'especially': 1297,
 'sticky': 3163,
 'situation': 2987,
 'quick': 2604,
 'dry': 1187,
 'lightweight': 1988,
 'underwear': 3535,
 'minimal': 2161,
 'space': 3088,
 'travel': 3455,
 'pack': 2363,
 'expose': 1328,
 'brush': 652,
 'waistband': 3647,
 'offer': 2303,
 'skin': 2999,
 'softness': 3068,
 'panel': 2381,
 'construction': 907,
 'traditional': 3438,
 'classic': 796,
 'fit': 1410,
 'functional': 1512,
 'fly': 1452,
 'recycled': 2655,
 'polyester': 2503,
 'moisture': 2184,
 'wicke': 3722,
 'performance': 2434,
 'inseam': 1818,
 'size': 2988,
 'silky': 2966,
 'capilene': 707,
 'ultralight': 3516,
 'breathable': 630,
 'elastic': 1231,
 'comfort': 855,
 'gladiodor': 1554,
 'natural': 2231,
 'odor': 2302,
 'control': 920,
 'garment': 1527,
 '99': 293,
 'mexico': 2132,
 'glory': 1563,
 'require': 2715,
 'movement': 2205,
 'decide': 1044,
 'poach': 2494,
 'route': 2803,
 'form': 1475,
 'active': 318,
 'sport'

In [21]:
vectorizer.get_feature_names_out()

array(['000', '03', '10', ..., 'zipping', 'zips', 'zone'],
      shape=(3792,), dtype=object)

# apply truncated SVD

In [22]:
svd_model = TruncatedSVD(n_components=10, n_iter=100, random_state=122) # try extract 10 topics
lsa = svd_model.fit_transform(X)
lsa # U matrix

array([[ 0.22053878, -0.09322154,  0.1908239 , ..., -0.00606485,
        -0.1596436 ,  0.0344486 ],
       [ 0.26135668, -0.09069683,  0.08568709, ..., -0.00542868,
        -0.12824351, -0.05113946],
       [ 0.22676998, -0.11088626,  0.08738371, ...,  0.06408708,
        -0.04861348, -0.01106634],
       ...,
       [ 0.30209059, -0.22912478, -0.12006132, ..., -0.20014047,
         0.05971854,  0.06708331],
       [ 0.25020992,  0.06758565, -0.08740051, ...,  0.01432163,
        -0.09067676,  0.01995356],
       [ 0.29106687,  0.06312237, -0.14543484, ...,  0.01332584,
        -0.08523144,  0.01565268]], shape=(500, 10))

Plus la valeur absolue est importante plus le document est important pour le topic. On peut filtrer les valeurs négatives.

In [23]:
topic_encoded_df = pd.DataFrame(lsa, columns = [f"topic_{i+1}"for i in range(10)], index = df.index)
topic_encoded_df

,topic_1,topic_2,topic_3,topic_4,topic_5,topic_6,topic_7,topic_8,topic_9,topic_10
0,0.220539,-0.093222,0.190824,0.069268,-0.116567,-0.011825,-0.068977,-0.006065,-0.159644,0.034449
1,0.261357,-0.090697,0.085687,0.097155,-0.020328,0.108230,-0.097411,-0.005429,-0.128244,-0.051139
2,0.226770,-0.110886,0.087384,0.018680,-0.011611,0.117567,0.007850,0.064087,-0.048613,-0.011066
3,0.360306,-0.244678,-0.147497,-0.158974,0.037464,-0.026077,-0.035463,-0.036631,-0.093804,-0.095214
4,0.389916,-0.260111,-0.062412,-0.219749,-0.032047,-0.024025,0.036399,0.046349,-0.011253,-0.161872
...,...,...,...,...,...,...,...,...,...,...
495,0.279584,-0.114605,0.391325,0.044450,-0.247390,-0.050409,-0.032006,-0.066400,-0.351867,0.095569
496,0.305781,-0.118931,0.488876,-0.030739,-0.299811,-0.072419,0.043531,0.018791,-0.240546,0.133462
497,0.302091,-0.229125,-0.120061,-0.248712,0.008869,-0.160892,0.088527,-0.200140,0.059719,0.067083
498,0.250210,0.067586,-0.087401,0.347759,0.016308,-0.152912,-0.118362,0.014322,-0.090677,0.019954


In [24]:
# m = vectorizer.get_feature_names_out()
# topic_encoded_df["keywords"] = [", ".join(m[r.toarray().ravel().argsort()[-5:][::-1]]) for r in X]

In [25]:
topic_encoded_df["keywords"] = df['nlp']
topic_encoded_df

,topic_1,topic_2,topic_3,topic_4,topic_5,topic_6,topic_7,topic_8,topic_9,topic_10,keywords
0,0.220539,-0.093222,0.190824,0.069268,-0.116567,-0.011825,-0.068977,-0.006065,-0.159644,0.034449,s reason boxer cult favorite cool especially s...
1,0.261357,-0.090697,0.085687,0.097155,-0.020328,0.108230,-0.097411,-0.005429,-0.128244,-0.051139,skin glory require movement boxer decide poach...
2,0.226770,-0.110886,0.087384,0.018680,-0.011611,0.117567,0.007850,0.064087,-0.048613,-0.011066,superbreathable fly brief minimalist s choice ...
3,0.360306,-0.244678,-0.147497,-0.158974,0.037464,-0.026077,-0.035463,-0.036631,-0.093804,-0.095214,skin climb ice switch rock traverse knife edge...
4,0.389916,-0.260111,-0.062412,-0.219749,-0.032047,-0.024025,0.036399,0.046349,-0.011253,-0.161872,high ridge steep ice alpine jacket serve true ...
...,...,...,...,...,...,...,...,...,...,...,...
495,0.279584,-0.114605,0.391325,0.044450,-0.247390,-0.050409,-0.032006,-0.066400,-0.351867,0.095569,cut loose madden crowd search undone capilene ...
496,0.305781,-0.118931,0.488876,-0.030739,-0.299811,-0.072419,0.043531,0.018791,-0.240546,0.133462,crew take edge fickle weather clearing conditi...
497,0.302091,-0.229125,-0.120061,-0.248712,0.008869,-0.160892,0.088527,-0.200140,0.059719,0.067083,need use morning time umbrella time handsome m...
498,0.250210,0.067586,-0.087401,0.347759,0.016308,-0.152912,-0.118362,0.014322,-0.090677,0.019954,wear cargo short bask glory sweat stain paint ...


In [26]:
# Assign each document to the topic it is the most linked to
topic_encoded_df["class_pred"] = [np.argmax(doc) for doc in lsa]
topic_encoded_df["class_pred"].value_counts()

class_pred
0    242
1     61
2     42
3     34
7     29
5     26
6     25
8     21
4     12
9      8
Name: count, dtype: int64

Le topic 1 récupère beaucoup trop de valeurs par rapport aux autres.

In [27]:
df.head(1)

,nlp
0,s reason boxer cult favorite cool especially s...


In [28]:
topic_encoded_df.dtypes

topic_1       float64
topic_2       float64
topic_3       float64
topic_4       float64
topic_5       float64
topic_6       float64
topic_7       float64
topic_8       float64
topic_9       float64
topic_10      float64
keywords       object
class_pred      int64
dtype: object

# print the confusion matrix for the topic against the target variable

On représente les topics dans un scatterplot. Comment représenter 10 topics ? Plus un point est proche d'un axe plus il est proche du topic (métrique distance cosinus)

In [31]:
topic_encoded_df.dtypes

topic_1       float64
topic_2       float64
topic_3       float64
topic_4       float64
topic_5       float64
topic_6       float64
topic_7       float64
topic_8       float64
topic_9       float64
topic_10      float64
keywords       object
class_pred      int64
target         object
dtype: object

In [ ]:
# cols = ['target', 'class_pred']
# topic_encoded_df[cols] = topic_encoded_df[cols].astype(str)

# évaluer le truncateSVD

Pour générer la heatmap, nous avons besoin de class_pred

In [ ]:
# # Create heatmap with plotly
# import plotly.figure_factory as ff
# from sklearn.metrics import confusion_matrix

# cm = confusion_matrix(y_true = topic_encoded_df["target"], y_pred = topic_encoded_df["class_pred"]) / len(topic_encoded_df)

In [36]:
cm

array([[0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   ,
        0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   ,
        0.   , 0.   ],
       [0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   ,
        0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   ,
        0.   , 0.   ],
       [0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   ,
        0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   ,
        0.   , 0.   ],
       [0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   ,
        0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   ,
        0.   , 0.   ],
       [0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   ,
        0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   ,
        0.   , 0.   ],
       [0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   ,
        0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   ,
        0.   , 0.   ],
       [0.

In [ ]:
# fig = ff.create_annotated_heatmap(cm.round(2), x = [i for i in range(20)], y = [i for i in range(20)])
# fig.update_layout(width = 1200)

# fig.show()